Note:
- This notebook is part of the Gao et al. (2026, in preparation) analysis workspace:
  Gao et al., Minute-scale coupling of chromatin marks and transcriptional bursts (2026, in preparation)
- For data layout, execution order, and key parameters, see the README.md in this folder:
  - analyses/02_Cross_correlation/README.md
- State labels:
  - Active = ON
  - Inactive = OFF
  CSV exports use Active/Inactive. Internal ON/OFF aliases and legacy filenames are retained only for compatibility.


# HDAC inhibitor time-lapse analysis (MCP / mTetR)

This notebook reproduces the analysis pipeline for HDAC inhibitor time-lapse movies described in the manuscript section:
"Analysis of HDAC inhibitor time-lapse data".

Input:
- Two-channel TIFF movies (time × channel × y × x)
  - channel 0: MCP (nascent transcription reporter)
  - channel 1: mTetR (locus marker)
- Optional per-frame cell label images (time × y × x), stored as: <movie_stem>_labels.tif

Output:
- Per-frame Active/Inactive (Active/Inactive) calls per cell
- Per-cell burst metrics (Ton, Toff, burst frequency) estimated with censoring-aware MLE
- Run-level tables (Active/Inactive durations with censoring flags)
- Figure-ready plots (state raster, duty cycle, Kaplan–Meier curves)

Note:
Raw movies are not included in this repository. You must provide local data paths.


## 1. Input data layout

Set ROOT_DIR to a folder that contains time-lapse movies (*.tif). Label masks are optional.

Movies:
- Location: ROOT_DIR/*.tif  (excluding *_labels.tif)
- Expected shape: (T, 2, Y, X)

Optional label masks (per frame):
- File name: <movie_stem>_labels.tif
- Expected shape: (T, Y, X)
- Each pixel stores a cell ID (0 = background)

Condition labels:
Movies are assigned to conditions using the filename prefix:
- "1-" -> DMSO
- "2-" -> TSA
- "3-" -> RGFP966


## 2. Key analysis parameters (report these for reproducibility)

Imaging:
- Frame interval (DELTA_MIN): 1 min
- Pixel size (PIXEL_SIZE_XY_NM): 130 nm

Spot detection (normalized images):
- Detector: Big-FISH detect_spots (recommended / required for reproducibility)
- Normalized threshold (THRESH_NORM): 1.25
- Spot radius (SPOT_RADIUS_XY_NM): 200 nm

Tracking / linking:
- Max displacement between consecutive frames (MAX_DISPLACEMENT_PX): 3 px (≈ 390 nm)
- Minimum consecutive frames required (MIN_TRACK_CONSEC): 3

Important:
If Big-FISH is not installed, this notebook can fall back to a simplified detector.
Results may differ substantially from the paper.


## 3. State calling (Active / Inactive) and censoring

Per frame, a cell is classified as Active if:
- An MCP spot is detected within MAX_DISPLACEMENT_PX of the locus position.

Locus position:
- Defined by the mTetR trajectory (one representative mTetR spot per cell per frame).
- If mTetR is missing in a frame, the locus position is inferred from neighboring frames
  using nearest-neighbor linking, and the same proximity criterion is applied.

Noise suppression:
- Single-frame Active calls without temporal support from adjacent frames are relabeled as Inactive.

Run extraction and censoring:
- From the binary time series, contiguous Active and Inactive runs are extracted.
- Runs touching the first frame are marked as left-censored.
- Runs touching the last frame are marked as right-censored.
- Dwell-time distributions are analyzed using Kaplan–Meier curves
  (exclude left-censored runs; treat right-censored runs as censored observations).


## 4. Outputs (saved files)

Output folder:
- ../outputs/<Gene>/tetr_mcp_hybrid_analysis/

Per movie:
- <movie_stem>_perframe.csv
  Frame-level table (cell_id, frame, on_off, spot positions, intensity, etc.)
- <movie_stem>_kinetics.csv
  Cell-level kinetics (alpha, beta, Ton, Toff, burst frequency)
- <movie_stem>_spots_debug.tif  (if SAVE_DEBUG_IMAGES = True)
  Debug stack with detected spots circled

Aggregated tables:
- aggregated_perframe.csv
- aggregated_kinetics.csv
- aggregated_cell_intensities.csv  (mean intensity per cell on Active frames)
- aggregated_runs.csv              (run lengths + censoring flags)
- aggregated_runs_by_cell.csv      (run table segmented per cell; used for KM plots)


## 5. How to run this notebook

1) Run the analysis cell for each gene / locus (e.g., Nanog, Sox2)
   - Set ROOT_DIR and parameters
   - This generates aggregated_perframe.csv and aggregated_runs.csv in outputs

2) Run the "runs_by_cell" cell
   - This creates aggregated_runs_by_cell.csv (recommended for survival analysis)

3) Run the plotting section
   - Generates PDF panels (raster, duty cycle, Kaplan–Meier, QC tables)
   - Outputs are written under figures/ (or a configured FIGDIR)


In [ ]:
# CSV boundaries accept legacy inputs and export publication state labels.
from pathlib import Path as _StatePath
import sys as _state_sys
_state_start = _StatePath.cwd().resolve()
_state_repo = next((p for p in (_state_start, *_state_start.parents)
                    if (p / "gao2026_state_labels.py").is_file()), None)
if _state_repo is None:
    raise RuntimeError("Run this notebook from within the Gao2026 code repository")
if str(_state_repo) not in _state_sys.path:
    _state_sys.path.insert(0, str(_state_repo))
from gao2026_state_labels import export_transcription_states, read_legacy_state_csv

# This experiment investigates transcriptional dynamics in mESCs under 
# various Epigenetic/HDAC inhibitor conditions:
# 1. DMSO (Control): Vehicle control to establish baseline transcriptional activity.
# 2. TSA: Trichostatin A treatment (Broad-spectrum HDAC inhibitor).
# 3. RGFP966: HDAC3-specific inhibitor treatment.


# Cell 1: Data Processing and Analysis Logic
# 1. Image Pre-processing: Loads 2-channel TIFFs (MCP/TETR) and associated cell labels.
# 2. Normalization & Detection: Uses robust_bg_norm and nucleus_norm_image for 
#    background correction, then detect_spots_norm (BigFISH) for spot identification.
# 3. Tracking & Linking: Implements select_single_spot_per_cell to link spots across 
#    frames within individual cell masks using distance-based thresholds.
# 4. Quantification: Measures transcriptional burst intensity via 2D Gaussian fitting 
#    (measure_spot_intensity_gaussian) on normalized image patches.
# 5. Kinetics Estimation: Calculates On/Off rates and burst frequency using Maximum 
#    Likelihood Estimation (mle_rate_discrete) while accounting for censored data.
# 6. Aggregation: Consolidates per-frame, kinetic, and intensity data across all cells 
#    and experimental groups into final CSV reports.

import os
from pathlib import Path
from glob import glob
import warnings
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import tifffile
from scipy.optimize import minimize_scalar
from scipy.optimize import curve_fit
from scipy import stats as scistats
from skimage.draw import circle_perimeter

warnings.filterwarnings("ignore")


# Configurable Parameters

DATASETS = [
    ("Nanog", "../../../data/03_HDAC_inhibitor/Nanog"),
    ("Sox2", "../../../data/03_HDAC_inhibitor/Sox2"),
]
SAVE_DEBUG_IMAGES = True  # Switch to save debug images with spots circled

# Imaging and detection parameters
PIXEL_SIZE_XY_NM    = 130.0
SPOT_RADIUS_XY_NM   = 200.0
THRESH_MCP  = 1.25
THRESH_TETR = 1.10
MAX_DISPLACEMENT_PX = 3.0  
MIN_TRACK_CONSEC    = 3
DELTA_MIN           = 1.0
TETR_RESCUE_RADIUS_PX = 6
TETR_RESCUE_THR       = THRESH_TETR * 0.85

# Define a fixed order for plotting groups
GROUP_ORDER = ["DMSO", "TSA", "RGFP966"]


# BigFISH import (Requires installation: pip install big-fish)

try:
    import bigfish.detection as bf_detect
except ImportError as e:
    raise ImportError("bigfish not found. Please install with: pip install big-fish") from e


# Helper Functions (Identical to original script logic)


def gaussian_2d(xy, amp, x0, y0, sx, sy, theta):
    x, y = xy
    x0, y0 = float(x0), float(y0)
    a = (np.cos(theta)**2)/(2*sx**2) + (np.sin(theta)**2)/(2*sy**2)
    b = -(np.sin(2*theta))/(4*sx**2) + (np.sin(2*theta))/(4*sy**2)
    c = (np.sin(theta)**2)/(2*sx**2) + (np.cos(theta)**2)/(2*sy**2)
    g = amp * np.exp( - (a*((x-x0)**2) + 2*b*(x-x0)*(y-y0) + c*((y-y0)**2)))
    return g.ravel()

def measure_spot_intensity_gaussian(img: np.ndarray, yx: Tuple[float, float], patch_size: int = 9) -> float:
    """Fits a 2D Gaussian on (already normalized) image patch; returns amplitude-integrated intensity (relative units)."""
    if patch_size % 2 == 0: patch_size += 1
    h, w = img.shape
    y, x = int(round(yx[0])), int(round(yx[1]))
    s = patch_size // 2

    if y - s < 0 or y + s + 1 > h or x - s < 0 or x + s + 1 > w:
        return np.nan

    patch = img[y-s : y+s+1, x-s : x+s+1].astype(np.float64)
    Y, X = np.mgrid[:patch_size, :patch_size]
    amp0 = np.max(patch)
    initial_guess = (max(amp0, 0.0), s, s, 1.5, 1.5, 0.0)
    try:
        popt, _ = curve_fit(gaussian_2d, (X.ravel(), Y.ravel()), patch.ravel(), p0=initial_guess,
                            bounds=([0, 0, 0, 0.5, 0.5, -np.pi], [np.inf, patch_size, patch_size, s, s, np.pi]))
        amp, x0, y0, sx, sy, _ = popt
        intensity = 2 * np.pi * amp * sx * sy
        return intensity if np.isfinite(intensity) else np.nan
    except (RuntimeError, ValueError):
        return np.nan

def robust_bg_norm(img2d: np.ndarray) -> Tuple[np.ndarray, float, float]:
    """Robust background normalization using median absolute deviation (fallback without label)"""
    flat = img2d.reshape(-1)
    med = np.median(flat)
    mad = np.median(np.abs(flat - med))
    bg_std = (mad / 0.6745) if mad > 0 else max(np.std(flat), 1e-6)
    inorm = (img2d - med) / max(bg_std, 1e-6)
    return inorm.astype(np.float32), float(med), float(bg_std)

def nucleus_norm_image(img2d: np.ndarray, labels_2d: Optional[np.ndarray], cell_id: Optional[int]) -> Tuple[np.ndarray, float, float]:
    """Normalizes by 'intranuclear mean/robust SD'; if no label, falls back to global robust normalization."""
    if labels_2d is None or cell_id is None or cell_id <= 0 or (labels_2d==cell_id).sum() == 0:
        return robust_bg_norm(img2d)
    mask = (labels_2d == cell_id)
    vals = img2d[mask].astype(np.float64)
    if vals.size < 10:
        return robust_bg_norm(img2d)
    mu  = float(np.mean(vals))
    mad = float(np.median(np.abs(vals - mu)))
    sigma = (mad / 0.6745) if mad > 0 else max(float(np.std(vals)), 1e-6)
    inorm = (img2d - mu) / max(sigma, 1e-6)
    return inorm.astype(np.float32), mu, sigma

def detect_spots_norm(inorm: np.ndarray, thr: float) -> np.ndarray:
    """Detect spots using BigFISH (threshold detection on normalized image)"""
    spots = bf_detect.detect_spots(
        images=inorm, threshold=thr, return_threshold=False,
        voxel_size=(PIXEL_SIZE_XY_NM, PIXEL_SIZE_XY_NM),
        spot_radius=(SPOT_RADIUS_XY_NM, SPOT_RADIUS_XY_NM)
    )
    return spots.astype(np.float32) if spots is not None else np.empty((0, 2), dtype=np.float32)

def classify_group(stem: str) -> str:
    if stem.startswith("1-"): return "DMSO"
    if stem.startswith("2-"): return "TSA"
    if stem.startswith("3-"): return "RGFP966"
    return "Unknown"

def load_label_stack_for(tiff_path: Path):
    lab_path = tiff_path.with_name(f"{tiff_path.stem}_labels.tif")
    if not lab_path.exists(): return None
    labs = tifffile.imread(str(lab_path))
    if labs.ndim != 3: raise ValueError(f"Labels shape {labs.shape} expected (T,Y,X)")
    return labs.astype(np.int32)

def split_spots_by_cell(spots_yx: np.ndarray, labels_2d: np.ndarray) -> Dict[int, np.ndarray]:
    out = {}
    H, W = labels_2d.shape
    for y, x in spots_yx:
        yi, xi = int(np.clip(round(y), 0, H-1)), int(np.clip(round(x), 0, W-1))
        cid = int(labels_2d[yi, xi])
        if cid > 0:
            out.setdefault(cid, []).append([float(y), float(x)])
    return {k: np.asarray(v, dtype=np.float32) for k, v in out.items()}

def find_closest_spot(spots: np.ndarray, ref_yx: Tuple[float, float], max_dist: float) -> Optional[Tuple[float, float]]:
    if spots is None or len(spots) == 0 or ref_yx is None: return None
    dists = np.linalg.norm(spots - np.array(ref_yx), axis=1)
    min_idx = np.argmin(dists)
    return tuple(spots[min_idx]) if dists[min_idx] <= max_dist else None

def select_single_spot_per_cell(tetr_spots: np.ndarray, mcp_spots: np.ndarray, prev_tetr_yx=None, prev_mcp_yx=None):
    selected_tetr, selected_mcp = None, None
    if prev_tetr_yx: selected_tetr = find_closest_spot(tetr_spots, prev_tetr_yx, MAX_DISPLACEMENT_PX)
    if prev_mcp_yx: selected_mcp = find_closest_spot(mcp_spots, prev_mcp_yx, MAX_DISPLACEMENT_PX)
    if selected_tetr and not selected_mcp: selected_mcp = find_closest_spot(mcp_spots, selected_tetr, MAX_DISPLACEMENT_PX)
    elif selected_mcp and not selected_tetr: selected_tetr = find_closest_spot(tetr_spots, selected_mcp, MAX_DISPLACEMENT_PX)
    if not selected_tetr and not selected_mcp and len(tetr_spots) > 0 and len(mcp_spots) > 0:
        dists = np.linalg.norm(tetr_spots[:, np.newaxis, :] - mcp_spots[np.newaxis, :, :], axis=2)
        min_dist = np.min(dists)
        if min_dist <= MAX_DISPLACEMENT_PX:
            t_idx, m_idx = np.unravel_index(np.argmin(dists), dists.shape)
            selected_tetr, selected_mcp = tuple(tetr_spots[t_idx]), tuple(mcp_spots[m_idx])
    if not selected_tetr and not prev_tetr_yx and len(tetr_spots) > 0 and not selected_mcp: selected_tetr = tuple(tetr_spots[0])
    if not selected_mcp and not prev_mcp_yx and not prev_tetr_yx and len(mcp_spots) > 0 and not selected_tetr: selected_mcp = tuple(mcp_spots[0])
    return selected_tetr, selected_mcp

def extract_runs_with_censoring(binary: np.ndarray) -> Dict[str, List]:
    if len(binary) == 0: return dict(on_lengths=[], on_left_censored=[], on_right_censored=[], off_lengths=[], off_left_censored=[], off_right_censored=[])
    T = len(binary)
    runs, cur, start = [], binary[0], 0
    for i in range(1, T):
        if binary[i] != cur:
            runs.append((cur, start, i-1)); cur, start = binary[i], i
    runs.append((cur, start, T-1))
    on_L, on_lc, on_rc, off_L, off_lc, off_rc = [], [], [], [], [], []
    for v, s, e in runs:
        L, lc, rc = e - s + 1, (s == 0), (e == T-1)
        (on_L if v == 1 else off_L).append(L); (on_lc if v == 1 else off_lc).append(lc); (on_rc if v == 1 else off_rc).append(rc)
    return dict(on_lengths=on_L, on_left_censored=on_lc, on_right_censored=on_rc, off_lengths=off_L, off_left_censored=off_lc, off_right_censored=off_rc)

def nll_discrete_exp(lmbda: float, lengths: List[int], cens: List[bool], delta: float) -> float:
    if lmbda <= 0: return np.inf
    ll = 0.0
    for k, is_cens in zip(lengths, cens):
        if k <= 0: continue
        ll += np.log(np.exp(-lmbda*delta*(k-1)) + 1e-300) if is_cens else np.log(np.exp(-lmbda*delta*(k-1)) - np.exp(-lmbda*delta*k) + 1e-300)
    return -ll

def mle_rate_discrete(lengths: List[int], left_cens: List[bool], right_cens: List[bool], delta: float) -> Tuple[float, Dict]:
    if not lengths: return np.nan, {"n_exact":0, "n_cens":0}
    C = [lc or rc for lc, rc in zip(left_cens, right_cens)]
    L_exact = [k for k, c in zip(lengths, C) if not c]
    ub = 10.0
    if L_exact and np.mean(L_exact) > 0: ub = max(2.0 / (np.mean(L_exact) * delta), 1.0)
    res = minimize_scalar(lambda lam: nll_discrete_exp(lam, lengths, C, delta), bounds=(1e-6, ub), method="bounded")
    return float(res.x), {"n_exact": int(sum(not c for c in C)), "n_cens": int(sum(c for c in C))}

def burst_freq(Ton: float, Toff: float) -> float:
    if np.any(np.isnan([Ton, Toff])): return np.nan
    s = Ton + Toff
    return (1.0 / s) if s > 0 else np.nan

def rescue_spot_from_window(inorm: np.ndarray,
                            center_yx: Tuple[float, float],
                            radius_px: int,
                            thr: float) -> Optional[Tuple[float, float]]:
    """Local max search in window around center; return (y,x) if max >= thr."""
    if center_yx is None:
        return None
    H, W = inorm.shape
    cy, cx = int(round(center_yx[0])), int(round(center_yx[1]))
    y0, y1 = max(0, cy - radius_px), min(H, cy + radius_px + 1)
    x0, x1 = max(0, cx - radius_px), min(W, cx + radius_px + 1)
    if y0 >= y1 or x0 >= x1:
        return None
    patch = inorm[y0:y1, x0:x1]
    maxv = float(np.max(patch))
    if maxv < thr:
        return None
    iy, ix = np.unravel_index(np.argmax(patch), patch.shape)
    return (float(y0 + iy), float(x0 + ix)) #####added

# Core: Process a single TIFF

def process_tiff_file(tiff_path: Path, out_dir: Path) -> None:
    """Processes a single TIFF file and saves per-file CSV results into out_dir."""
    arr = tifffile.imread(str(tiff_path))
    if arr.ndim != 4 or arr.shape[1] != 2: raise ValueError(f"TIFF shape {arr.shape} not (T, 2, Y, X).")
    T, C, Y, X = arr.shape
    MCP, TETR = 0, 1
    labels = load_label_stack_for(tiff_path)
    if labels is not None and labels.shape[0] != T: raise ValueError(f"Label T mismatch: {labels.shape[0]} vs {T}")

    mcp_raw_cache, tetr_raw_cache, mcp_spots_cache, tetr_spots_cache = [], [], [], []
    for t in range(T):
        mcp_raw, tetr_raw = arr[t, MCP].astype(np.float32), arr[t, TETR].astype(np.float32)
        mcp_norm, _, _ = robust_bg_norm(mcp_raw)
        tetr_norm, _, _ = robust_bg_norm(tetr_raw)
        mcp_raw_cache.append(mcp_raw)
        tetr_raw_cache.append(tetr_raw)
        mcp_spots_cache.append(detect_spots_norm(mcp_norm, THRESH_MCP))
        tetr_spots_cache.append(detect_spots_norm(tetr_norm, THRESH_TETR))

    cell_ids = [1]
    if labels is not None: cell_ids = sorted({int(x) for t in range(T) for x in np.unique(labels[t]) if x > 0})
    
    cell_selected_spots = {cid: {'tetr': {}, 'mcp': {}} for cid in cell_ids}
    for cid in cell_ids:
        prev_tetr, prev_mcp = None, None
        for t in range(T):
            labs2d = labels[t] if labels is not None else np.ones((Y,X), dtype=int)
            tetr_in_cell = split_spots_by_cell(tetr_spots_cache[t], labs2d).get(cid, np.empty((0,2)))
            mcp_in_cell = split_spots_by_cell(mcp_spots_cache[t], labs2d).get(cid, np.empty((0,2)))
            sel_tetr, sel_mcp = select_single_spot_per_cell(tetr_in_cell, mcp_in_cell, prev_tetr, prev_mcp)
            if sel_tetr: cell_selected_spots[cid]['tetr'][t] = sel_tetr; prev_tetr = sel_tetr
            if sel_mcp: cell_selected_spots[cid]['mcp'][t] = sel_mcp; prev_mcp = sel_mcp

    for cid in cell_ids:
        prev_tetr = None
        for t in range(T):
            tetr_yx = cell_selected_spots[cid]['tetr'].get(t, None)
            mcp_yx  = cell_selected_spots[cid]['mcp'].get(t, None)
            labs2d = labels[t] if labels is not None else None

            tetr_norm_cell, _, _ = nucleus_norm_image(tetr_raw_cache[t], labs2d, cid)
            if tetr_yx is None:

                center = mcp_yx if mcp_yx is not None else prev_tetr
                tetr_yx = rescue_spot_from_window(
                    tetr_norm_cell, center_yx=center,
                    radius_px=TETR_RESCUE_RADIUS_PX,
                    thr=TETR_RESCUE_THR
                )

                if tetr_yx is None and (mcp_yx is not None) and (prev_tetr is not None):
                    tetr_yx = rescue_spot_from_window(
                        tetr_norm_cell, center_yx=prev_tetr,
                        radius_px=TETR_RESCUE_RADIUS_PX,
                        thr=TETR_RESCUE_THR
                    )
            if tetr_yx is not None:
                cell_selected_spots[cid]['tetr'][t] = tetr_yx
                prev_tetr = tetr_yx
            else:

                if prev_tetr is not None:
                    prev_tetr = prev_tetr  ###added

    rows = []
    for cid in cell_ids:
        for t in range(T):
            tetr_yx, mcp_yx = cell_selected_spots[cid]['tetr'].get(t), cell_selected_spots[cid]['mcp'].get(t)
            on_flag = 0
            if tetr_yx and mcp_yx and np.linalg.norm(np.array(tetr_yx) - np.array(mcp_yx)) <= MAX_DISPLACEMENT_PX: on_flag = 1
            elif mcp_yx: on_flag = 1
            rel_intensity = np.nan
            if on_flag and mcp_yx:
                labs2d = labels[t] if labels is not None else None
                mcp_norm_cell, _, _ = nucleus_norm_image(mcp_raw_cache[t], labs2d, cid)
                rel_intensity = measure_spot_intensity_gaussian(mcp_norm_cell, mcp_yx)
            rows.append(dict(file=tiff_path.name, group=classify_group(tiff_path.name), frame=t, cell_id=cid, on_off=on_flag, tetr_stable=bool(tetr_yx), mcp_y=mcp_yx[0] if mcp_yx else np.nan, mcp_x=mcp_yx[1] if mcp_yx else np.nan, mcp_integrated_intensity=rel_intensity))
    perframe = pd.DataFrame(rows).sort_values(["cell_id", "frame"]).reset_index(drop=True)

    final_dfs = []
    for cid, dfc in perframe.groupby("cell_id"):
        dfc = dfc.copy(); on_off = dfc['on_off'].values.copy()
        for i in range(len(dfc)):
            if not dfc['tetr_stable'].iloc[i] and dfc['on_off'].iloc[i] == 1:
                connected = (i > 0 and dfc['on_off'].iloc[i-1] == 1 and np.linalg.norm(dfc[['mcp_y','mcp_x']].iloc[i] - dfc[['mcp_y','mcp_x']].iloc[i-1]) <= MAX_DISPLACEMENT_PX) or \
                            (i < len(dfc)-1 and dfc['on_off'].iloc[i+1] == 1 and np.linalg.norm(dfc[['mcp_y','mcp_x']].iloc[i] - dfc[['mcp_y','mcp_x']].iloc[i+1]) <= MAX_DISPLACEMENT_PX)
                if not connected: on_off[i] = 0
        dfc['on_off'] = on_off
        final_dfs.append(dfc)
    if final_dfs: perframe = pd.concat(final_dfs, ignore_index=True)
    perframe.drop(columns=['mcp_y', 'mcp_x']).pipe(export_transcription_states).to_csv(out_dir / f"{tiff_path.stem}_perframe.csv", index=False)

    if SAVE_DEBUG_IMAGES:
        debug_stack = []
        for t in range(T):
            mcp_draw, tetr_draw = arr[t, MCP].copy(), arr[t, TETR].copy()
            mcp_v, tetr_v = np.percentile(mcp_draw, 99.9), np.percentile(tetr_draw, 99.9)
            for cid in cell_ids:
                if t in cell_selected_spots[cid]['tetr']: rr, cc = circle_perimeter(int(round(cell_selected_spots[cid]['tetr'][t][0])), int(round(cell_selected_spots[cid]['tetr'][t][1])), 5, shape=(Y,X)); tetr_draw[rr, cc] = tetr_v
                if t in cell_selected_spots[cid]['mcp']: rr, cc = circle_perimeter(int(round(cell_selected_spots[cid]['mcp'][t][0])), int(round(cell_selected_spots[cid]['mcp'][t][1])), 5, shape=(Y,X)); mcp_draw[rr, cc] = mcp_v
            debug_stack.append(np.stack([mcp_draw, tetr_draw], axis=0))
        tifffile.imsave(out_dir / f"{tiff_path.stem}_spots_debug.tif", np.array(debug_stack, dtype=np.float32), imagej=True)

    kin_rows = []
    for cid, dfc in perframe.groupby("cell_id", sort=True):
        if dfc['on_off'].sum() == 0 or len(dfc) < MIN_TRACK_CONSEC * 2: continue
        seq = dfc["on_off"].to_numpy()
        rr = extract_runs_with_censoring(seq)
        b_hat, _ = mle_rate_discrete(rr["on_lengths"], rr["on_left_censored"], rr["on_right_censored"], DELTA_MIN)
        a_hat, _ = mle_rate_discrete(rr["off_lengths"], rr["off_left_censored"], rr["off_right_censored"], DELTA_MIN)
        Ton = (1.0 / b_hat) if (b_hat > 0 and np.isfinite(b_hat)) else np.nan
        Toff = (1.0 / a_hat) if (a_hat > 0 and np.isfinite(a_hat)) else np.nan
        kin_rows.append(dict(file=tiff_path.name, group=classify_group(tiff_path.name), cell_id=cid, a_off_to_on=a_hat, b_on_to_off=b_hat, T_on_mean=Ton, T_off_mean=Toff, burst_frequency_per_min=burst_freq(Ton, Toff), frames_analyzed=len(seq)))
    if kin_rows: pd.DataFrame(kin_rows).pipe(export_transcription_states).to_csv(out_dir / f"{tiff_path.stem}_kinetics.csv", index=False)

def run_analysis_and_save_results(root_dir: str, gene: str):
    """Main function: Runs all analyses and saves aggregated results to CSV."""
    root   = Path(root_dir)
    outdir = Path.cwd().parent / "outputs" / gene / "tetr_mcp_hybrid_analysis"
    outdir.mkdir(parents=True, exist_ok=True)

    files = sorted([p for p in root.glob("*.tif") if not p.name.endswith("_labels.tif")])
    print(f"Starting processing of {len(files)} TIFF files...")
    for i, fp in enumerate(files, 1):
        print(f"  [{i}/{len(files)}] Processing {fp.name}...")
        try:
            process_tiff_file(fp, outdir)
        except Exception as e:
            print(f"  Error processing {fp.name}: {e}")

    print("\nAggregating all results...")
    perframe_files = list(outdir.glob("*_perframe.csv"))
    kinetics_files = list(outdir.glob("*_kinetics.csv"))
    
    if not perframe_files:
        print("No per-frame result files found. Exiting.")
        return None

    # 1. Aggregate per-frame data
    perframes = pd.concat([read_legacy_state_csv(f) for f in perframe_files], ignore_index=True)
    perframes.pipe(export_transcription_states).to_csv(outdir / "aggregated_perframe.csv", index=False)
    print(f"  Saved aggregated per-frame data to: {outdir / 'aggregated_perframe.csv'}")

    # 2. Aggregate kinetic data
    kinetics = pd.DataFrame()
    if kinetics_files:
        kinetics = pd.concat([read_legacy_state_csv(f) for f in kinetics_files if f.stat().st_size > 0], ignore_index=True)
    kinetics.pipe(export_transcription_states).to_csv(outdir / "aggregated_kinetics.csv", index=False)
    print(f"  Saved aggregated kinetic data to: {outdir / 'aggregated_kinetics.csv'}")
    
    # 3. Calculate and save mean intensities
    on_frames = perframes[perframes['on_off'] == 1].copy()
    if not on_frames.empty:
        cell_intensities = on_frames.groupby(['group', 'file', 'cell_id'])['mcp_integrated_intensity'].mean().reset_index()
        cell_intensities.rename(columns={'mcp_integrated_intensity': 'mean_intensity'}, inplace=True)
    else:
        cell_intensities = pd.DataFrame(columns=['group', 'file', 'cell_id', 'mean_intensity'])
    cell_intensities.pipe(export_transcription_states).to_csv(outdir / "aggregated_cell_intensities.csv", index=False)
    print(f"  Saved aggregated mean intensity data to: {outdir / 'aggregated_cell_intensities.csv'}")

    # 4. Extract and save all ON/OFF runs
    all_runs = []
    for (group, file, cell_id), dfc in perframes.groupby(['group', 'file', 'cell_id']):
        if len(dfc) < MIN_TRACK_CONSEC: continue
        rr = extract_runs_with_censoring(dfc['on_off'].to_numpy())
        for L, lc, rc in zip(rr['on_lengths'], rr['on_left_censored'], rr['on_right_censored']):
            all_runs.append({'group': group, 'kind': 'ON', 'length_frames': int(L), 'left_censored': bool(lc), 'right_censored': bool(rc)})
        for L, lc, rc in zip(rr['off_lengths'], rr['off_left_censored'], rr['off_right_censored']):
            all_runs.append({'group': group, 'kind': 'OFF', 'length_frames': int(L), 'left_censored': bool(lc), 'right_censored': bool(rc)})
    runs_all = pd.DataFrame(all_runs)
    runs_all.pipe(export_transcription_states).to_csv(outdir / "aggregated_runs.csv", index=False)
    print(f"  Saved aggregated ON/OFF run data to: {outdir / 'aggregated_runs.csv'}")

    print("\n=== Analysis Complete ===")
    print(f"All results are saved in: {outdir}")
    return outdir

def build_runs_by_cell(outdir: Path, delta_min: float):
    outdir = Path(outdir)
    PERFRAME_PATH = outdir / "aggregated_perframe.csv"

    if not PERFRAME_PATH.exists():
        raise FileNotFoundError(f"Not found: {PERFRAME_PATH} (run Cell 1 first)")

    perframes = read_legacy_state_csv(PERFRAME_PATH)

    required = {"group","file","cell_id","frame","on_off"}
    missing = required - set(perframes.columns)
    if missing:
        raise ValueError(f"aggregated_perframe.csv missing columns: {missing}")

    perframes["cell_uid"] = perframes["file"].astype(str) + "_c" + perframes["cell_id"].astype(str)
    perframes = perframes.sort_values(["group","cell_uid","frame"]).reset_index(drop=True)

    def extract_segments(binary: np.ndarray):
        """
        binary: 0/1 array
        returns list of dict: state(0/1), start_idx, end_idx, length, left_censored, right_censored
        """
        binary = np.asarray(binary).astype(int)
        T = len(binary)
        if T == 0:
            return []
        segs = []
        cur = binary[0]
        start = 0
        for i in range(1, T):
            if binary[i] != cur:
                end = i - 1
                segs.append((cur, start, end))
                cur = binary[i]
                start = i
        segs.append((cur, start, T - 1))

        out = []
        for state, s, e in segs:
            L = int(e - s + 1)
            out.append({
                "state": int(state),
                "start_frame": int(s),
                "end_frame": int(e),
                "length_frames": L,
                "left_censored": (s == 0),
                "right_censored": (e == T - 1),
            })
        return out

    rows = []
    for (grp, cell_uid), dfc in perframes.groupby(["group","cell_uid"], sort=False):
        seq = dfc["on_off"].astype(int).to_numpy()
        segs = extract_segments(seq)
        file_ = dfc["file"].iloc[0]
        cell_id_ = dfc["cell_id"].iloc[0]
        for seg in segs:
            rows.append({
                "group": grp,
                "file": file_,
                "cell_id": cell_id_,
                "cell_uid": cell_uid,
                "kind": "ON" if seg["state"] == 1 else "OFF",
                "start_frame": seg["start_frame"],
                "end_frame": seg["end_frame"],
                "length_frames": seg["length_frames"],
                "length_min": seg["length_frames"] * delta_min,
                "left_censored": bool(seg["left_censored"]),
                "right_censored": bool(seg["right_censored"]),
            })

    runs_cell = pd.DataFrame(rows)

    # Save (use this for subsequent analyses)
    RUNS_CELL_PATH = outdir / "aggregated_runs_by_cell.csv"
    runs_cell.pipe(export_transcription_states).to_csv(RUNS_CELL_PATH, index=False)
    print(f"[saved] {RUNS_CELL_PATH}")
    return runs_cell


for gene, root_dir in DATASETS:
    print(f"\n=== {gene} ===")
    outdir = run_analysis_and_save_results(root_dir, gene)
    if outdir is None:
        continue
    build_runs_by_cell(outdir, DELTA_MIN)

## Downstream visualizations (paper-ready panels)

This section generates:
- State raster plots (cells × time)
- Duty cycle (fraction of Active frames per cell)
- Kaplan–Meier curves for Active/Inactive durations
- QC summary tables (cells, runs, censoring fractions)

All plots are saved as PDF for direct use in figure assembly.


In [ ]:
# --- Imports & global plot style (Arial, PDF) ---

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# (optional) stats utilities
from scipy.stats import chi2

# ---- Output folders ----
FIGDIR = Path("../outputs/figs")
TABLEDIR = Path("../outputs/tables")
FIGDIR.mkdir(exist_ok=True, parents=True)
TABLEDIR.mkdir(exist_ok=True, parents=True)

# ---- Global plotting parameters (edit here) ----
PLOT_CFG = {
    # font sizes
    "font_base": 8,
    "font_axes_label": 9,
    "font_axes_title": 9,
    "font_ticks": 8,
    "font_legend": 7,

    # figure sizes (inches)
    "figsize_small": (3.3, 2.6),
    "figsize_medium": (4.0, 3.0),
    "figsize_wide": (6.0, 3.0),
    "dpi": 300,
}

def set_mpl_style(cfg=PLOT_CFG):
    """Set matplotlib rcParams (Arial + PDF text embedding)."""
    plt.rcParams.update({
        # font
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": cfg["font_base"],
        "axes.labelsize": cfg["font_axes_label"],
        "axes.titlesize": cfg["font_axes_title"],
        "xtick.labelsize": cfg["font_ticks"],
        "ytick.labelsize": cfg["font_ticks"],
        "legend.fontsize": cfg["font_legend"],

        # publication-friendly PDF/PS fonts (editable text)
        "pdf.fonttype": 42,
        "ps.fonttype": 42,

        # simple look
        "axes.spines.top": False,
        "axes.spines.right": False,
    })

set_mpl_style()
print("[OK] matplotlib style set. Output:", FIGDIR.resolve())


In [ ]:
# --- Paths to precomputed analysis outputs (edit if needed) ---

ROOT = Path.cwd()

NANOG_DIR = (ROOT.parent / "outputs" / "Nanog" / "tetr_mcp_hybrid_analysis").resolve()
SOX2_DIR  = (ROOT.parent / "outputs" / "Sox2"  / "tetr_mcp_hybrid_analysis").resolve()

print("NANOG_DIR:", NANOG_DIR)
print("SOX2_DIR :", SOX2_DIR)

for p in [NANOG_DIR, SOX2_DIR]:
    if not p.exists():
        print(f"[WARN] Not found: {p}")


In [ ]:
# --- Data loader (reuses aggregated CSVs) ---

def _ensure_cell_uid(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "cell_uid" not in df.columns:
        if ("file" in df.columns) and ("cell_id" in df.columns):
            df["cell_uid"] = df["file"].astype(str) + "_c" + df["cell_id"].astype(str)
        else:
            raise ValueError("Cannot construct cell_uid (need columns: file, cell_id).")
    return df

def load_dataset(gene: str, analysis_dir: Path) -> dict:
    analysis_dir = Path(analysis_dir)

    perframe_path = analysis_dir / "aggregated_perframe.csv"
    runs_by_cell_path = analysis_dir / "aggregated_runs_by_cell.csv"
    runs_path = analysis_dir / "aggregated_runs.csv"

    if not perframe_path.exists():
        raise FileNotFoundError(perframe_path)

    perframe = read_legacy_state_csv(perframe_path)
    perframe = _ensure_cell_uid(perframe)

    if runs_by_cell_path.exists():
        runs = read_legacy_state_csv(runs_by_cell_path)
    elif runs_path.exists():
        runs = read_legacy_state_csv(runs_path)
    else:
        raise FileNotFoundError(f"Neither {runs_by_cell_path.name} nor {runs_path.name} exists in {analysis_dir}")

    runs = _ensure_cell_uid(runs)

    # Basic sanity checks
    req_runs = {"group","kind","length_min","left_censored","right_censored","cell_uid"}
    missing = req_runs - set(runs.columns)
    if missing:
        raise ValueError(f"Missing columns in runs: {missing}")

    req_pf = {"group","frame","on_off","cell_uid"}
    missing_pf = req_pf - set(perframe.columns)
    if missing_pf:
        raise ValueError(f"Missing columns in perframe: {missing_pf}")

    return {
        "gene": gene,
        "dir": analysis_dir,
        "perframe": perframe,
        "runs": runs,
    }

datasets = []
for gene, p in [("Nanog", NANOG_DIR), ("Sox2", SOX2_DIR)]:
    try:
        datasets.append(load_dataset(gene, p))
        print(f"[OK] Loaded {gene}: perframe={len(datasets[-1]['perframe'])}, runs={len(datasets[-1]['runs'])}")
    except Exception as e:
        print(f"[SKIP] {gene}: {e}")

datasets


In [ ]:
# --- Group labels / ordering ---

# Minimal (main-figure) set
GROUP_ORDER_MAIN = ["DMSO", "TSA", "RGFP966"]

# Full set (ED)
GROUP_ORDER_FULL = ["DMSO", "TSA", "RGFP966"]

DISPLAY_NAME = {
    "RGFP966": "RGFP966",
}


KIND_LABEL = {
    "ON": "Active",
    "OFF": "Inactive"
}

def nice_group(g: str) -> str:
    return DISPLAY_NAME.get(g, g)


In [ ]:
# --- Survival analysis utilities (KM) ---

def kaplan_meier(times, events):
    """Kaplan–Meier estimator for right-censored data.

    Parameters
    ----------
    times : array-like (>=0)
    events : array-like (1=event observed, 0=right-censored)

    Returns
    -------
    t : np.ndarray
        Step times, starting with 0.
    S : np.ndarray
        Survival values S(t)=P(T>=t) in 'post' convention.
    """
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)

    m = np.isfinite(times) & (times >= 0) & np.isfinite(events)
    times = times[m]; events = events[m]

    if len(times) == 0:
        return np.array([0.0]), np.array([1.0])

    event_times = np.unique(times[events == 1])
    S = 1.0
    t_list = [0.0]
    s_list = [1.0]

    for t in event_times:
        n_risk = np.sum(times >= t)
        d = np.sum((times == t) & (events == 1))
        if n_risk <= 0:
            continue
        S *= (1.0 - d / n_risk)
        t_list.append(float(t))
        s_list.append(float(S))

    return np.array(t_list), np.array(s_list)

def prepare_runs(runs: pd.DataFrame, kind: str, groups, exclude_left=True):
    df = runs[(runs["kind"] == kind) & (runs["group"].isin(groups))].copy()
    if exclude_left:
        df = df[~df["left_censored"]].copy()
    df["event"] = (~df["right_censored"]).astype(int)
    df = df[np.isfinite(df["length_min"]) & (df["length_min"] > 0)].copy()
    return df

In [ ]:
# --- Plot helpers (save as PDF) ---

def savefig_pdf(fig, outpath: Path):
    outpath = Path(outpath)
    outpath.parent.mkdir(exist_ok=True, parents=True)
    fig.savefig(outpath, bbox_inches="tight", dpi=PLOT_CFG["dpi"])
    plt.close(fig)
    print("[saved]", outpath)

def plot_km_panel(runs: pd.DataFrame, kind: str, gene: str, groups, exclude_left=True,
                  logy=True, xlim=None, ylim=None, title=None, outname=None):

    df = prepare_runs(runs, kind=kind, groups=groups, exclude_left=exclude_left)

    fig, ax = plt.subplots(figsize=(3.33,2.5))

    for grp in groups:
        sub = df[df["group"] == grp]
        if sub.empty:
            continue
        t, S = kaplan_meier(sub["length_min"].values, sub["event"].values)
        ax.step(t, S, where="post",
                label=f"{nice_group(grp)} (runs={len(sub)})")

    # ---- Convert labels for display ----
    kind_label = KIND_LABEL.get(kind, kind)

    ax.set_xlabel(f"{kind_label} duration (min)")
    ax.set_ylabel("Survival S(t) = P(duration ≥ t)")

    if title is None:
        title = f"{gene}: Kaplan–Meier ({kind_label})"
    ax.set_title(title)

    if logy:
        ax.set_yscale("log")
        if ylim is None:
            ax.set_ylim(1e-3, 1.05)
    else:
        ax.set_ylim(0, 1.05)

    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)

    ax.grid(alpha=0.3)
    ax.legend(frameon=False)

    if outname is None:
        outname = f"{gene}_KM_{kind_label}_groups{'-'.join(groups)}.pdf"
    savefig_pdf(fig, FIGDIR / outname)






from scipy.stats import mannwhitneyu
import numpy as np

def holm_adjust_array(pvals):
    """Holm step-down adjustment for a list/array of p-values."""
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    adj = np.empty(m, dtype=float)

    running_max = 0.0
    for rank, idx in enumerate(order):
        adj_p = (m - rank) * pvals[idx]
        running_max = max(running_max, adj_p)
        adj[idx] = min(1.0, running_max)
    return adj

def p_to_stars(p):
    if p < 1e-3: return "***"
    if p < 1e-2: return "**"
    if p < 5e-2: return "*"
    return "ns"

def add_sig_bracket(ax, x1, x2, y, text, h=0.02, lw=0.8):
    """
    Draw a significance bracket between x1 and x2 at height y (data coords).
    h: vertical height of bracket (data coords)
    """
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], color="black", lw=lw, clip_on=False)
    ax.text((x1+x2)/2, y+h, text, ha="center", va="bottom", fontsize=PLOT_CFG.get("fontsize", 7))



def duty_cycle_pvals(dc: pd.DataFrame, base="DMSO", comps=("TSA","RGFP966")):
    """
    dc: output of duty_cycle_table (columns: group, cell_uid, duty_cycle)
    returns: list of dict with raw p and Holm-adjusted p
    """
    base_vals = dc.loc[dc["group"]==base, "duty_cycle"].dropna().values
    results = []

    raw_ps = []
    pairs = []
    for g in comps:
        vals = dc.loc[dc["group"]==g, "duty_cycle"].dropna().values
        if len(base_vals) < 2 or len(vals) < 2:
            p = np.nan
        else:
            p = mannwhitneyu(base_vals, vals, alternative="two-sided").pvalue
        raw_ps.append(p)
        pairs.append((base, g))

    adj_ps = holm_adjust_array(raw_ps)

    for (b, g), p, p_adj in zip(pairs, raw_ps, adj_ps):
        results.append({
            "compare": f"{b} vs {g}",
            "p_raw": float(p) if np.isfinite(p) else np.nan,
            "p_holm": float(p_adj) if np.isfinite(p_adj) else np.nan,
        })
    return results



def duty_cycle_table(perframe: pd.DataFrame, groups) -> pd.DataFrame:
    """
    perframe: expects at least {group, cell_uid, on_off} in columns
    duty_cycle = mean on_off per cell (= fraction of ON frames)
    """
    pf = perframe[perframe["group"].isin(groups)].copy()
    pf = _ensure_cell_uid(pf)

    # Treat on_off as numeric (assumed 0/1)
    pf = pf[np.isfinite(pf["on_off"])].copy()

    dc = (pf.groupby(["group", "cell_uid"], sort=False)["on_off"]
            .mean()
            .reset_index()
            .rename(columns={"on_off": "duty_cycle"}))
    return dc


def plot_duty_cycle(perframe: pd.DataFrame, gene: str, groups, outname=None, title=None):
    dc = duty_cycle_table(perframe, groups)
    fig, ax = plt.subplots(figsize=(1.2,2.3))

    data = [dc.loc[dc["group"]==g, "duty_cycle"].values for g in groups]
    ax.boxplot(data, positions=np.arange(len(groups)), widths=0.5, showfliers=False)

    # overlay points (jitter)
    rng = np.random.default_rng(0)
    for i, g in enumerate(groups):
        y = dc.loc[dc["group"]==g, "duty_cycle"].values
        x = rng.normal(i, 0.06, size=len(y))
        ax.plot(x, y, "o", markersize=3, alpha=0.6)

    # -------------------- p-value annotations (MW + Holm) --------------------
    # Here, compare TSA/RGFP966 against DMSO as the base (only if present)
    base = "DMSO"
    comps = [g for g in ["TSA", "RGFP966"] if g in groups]

    if base in groups and len(comps) > 0:
        stats = duty_cycle_pvals(dc, base=base, comps=tuple(comps))

        # Bracket height: add a little above the data maximum
        y_max = np.nanmax(dc["duty_cycle"].values)
        y = y_max + 0.03  # Adjust here for appearance (duty is 0–1, so a fixed offset is OK)
        step = 0.12       # Stagger to avoid overlapping multiple brackets

        # Use the index within groups for x positions
        g_to_x = {g:i for i, g in enumerate(groups)}

        for k, r in enumerate(stats):
            comp = r["compare"]  # "DMSO vs TSA"
            g = comp.split(" vs ")[1]
            x1 = g_to_x[base]
            x2 = g_to_x[g]

            p_adj = r["p_holm"]
            # Display format: show p-values or stars
            label = f"{p_to_stars(p_adj)}" if np.isfinite(p_adj) else "n/a"
            # If you want to show p-values directly:
            # label = f"p={p_adj:.3g}" if np.isfinite(p_adj) else "n/a"

            add_sig_bracket(ax, x1, x2, y + k*step, label, h=0.015, lw=0.8)

    # ------------------------------------------------------------------------

    ax.set_xticks(np.arange(len(groups)))
    ax.set_xticklabels([nice_group(g) for g in groups], rotation=30, ha="right")
    ax.set_ylabel("Fraction of Active frames")
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.3)

    if outname is None:
        outname = f"{gene}_duty_cycle_groups{'-'.join(groups)}.pdf"
    savefig_pdf(fig, FIGDIR / outname)
    return dc



def plot_state_raster(perframe: pd.DataFrame, gene: str, groups, n_cells=35,
                      frame_to_min=1.0, outname=None, title=None):
    """ON/OFF raster (cells × time) for each group, stacked vertically."""
    pf = perframe[perframe["group"].isin(groups)].copy()
    pf = _ensure_cell_uid(pf)

    # build matrices per group
    mats = []
    for g in groups:
        sub = pf[pf["group"]==g].copy()
        if sub.empty:
            mats.append(None)
            continue

        mat = sub.pivot_table(index="cell_uid", columns="frame", values="on_off", aggfunc="first")
        # sort cells by duty cycle (descending)
        mat = mat.loc[mat.mean(axis=1).sort_values(ascending=False).index]
        mat = mat.iloc[:n_cells]
        mats.append(mat)

    # determine common x-axis
    max_frame = 0
    for m in mats:
        if m is not None and m.shape[1] > max_frame:
            max_frame = m.shape[1]
    t = np.arange(max_frame) * frame_to_min

    fig_h = 0.8*len(groups) + 1.2
    #fig, axes = plt.subplots(nrows=len(groups), ncols=1, figsize=(PLOT_CFG["figsize_wide"][0], fig_h), sharex=True)
    fig, axes = plt.subplots(nrows=len(groups), ncols=1, figsize=(2.0, 4.0), sharex=True)
    if len(groups) == 1:
        axes = [axes]

    for ax, g, mat in zip(axes, groups, mats):
        if mat is None:
            ax.axis("off")
            continue
        # reindex columns to full range
        mat2 = mat.reindex(columns=np.arange(max_frame))
        from matplotlib.colors import ListedColormap

        cmap = ListedColormap(["lightgray", "red"])  # OFF=0, ON=1
        img = ax.imshow(mat2.values, aspect="auto", interpolation="nearest",
                        vmin=0, vmax=1, cmap=cmap)
        
        # --- add horizontal white separators between cells (y only) ---
        n_rows = mat2.shape[0]
        ax.set_yticks(np.arange(-0.5, n_rows, 1), minor=True)   # Row boundaries
        ax.grid(which="minor", axis="y", color="white", linewidth=0.3)
        ax.tick_params(which="minor", left=False, right=False)  # Don't show minor ticks
        
        #img = ax.imshow(mat2.values, aspect="auto", interpolation="nearest", vmin=0, vmax=1)
        ax.set_ylabel(nice_group(g))
        #ax.set_yticks([])

    axes[-1].set_xlabel("Time (min)")
    # tick every 10 min
    if max_frame > 0:
        step = int(10/frame_to_min) if frame_to_min > 0 else 10
        xticks = np.arange(0, max_frame, step=max(1, step))
        axes[-1].set_xticks(xticks)
        axes[-1].set_xticklabels([str(int(x*frame_to_min)) for x in xticks])

    # if title is None:
    #     title = f"{gene}: ON/OFF state raster (top {n_cells} cells / group)"
    fig.suptitle(title, y=1.02)

    if outname is None:
        outname = f"{gene}_state_raster_groups{'-'.join(groups)}.pdf"
    savefig_pdf(fig, FIGDIR / outname)

In [ ]:
# --- Fig. 5 main candidates: raster, duty cycle ---

for ds in datasets:
    gene = ds["gene"]
    runs = ds["runs"]
    perframe = ds["perframe"]

    print("\n=== ", gene, "===")

    # (1) state raster
    plot_state_raster(perframe, gene=gene, groups=GROUP_ORDER_MAIN, n_cells=100,
                      outname=f"Fig5_{gene}_state_raster_MAIN.pdf")

    # (2) duty cycle
    dc = plot_duty_cycle(perframe, gene=gene, groups=GROUP_ORDER_MAIN,
                         outname=f"Fig5_{gene}_duty_cycle_MAIN.pdf")
    dc.pipe(export_transcription_states).to_csv(TABLEDIR / f"{gene}_duty_cycle_celllevel_MAIN.csv", index=False)

This section produces more detailed plots and QC summaries that are typically better suited for **Extended Data**:

- Full Kaplan–Meier curves for **Active** and **Inactive** durations (all 4 groups).
- QC table: number of cells, number of runs, censoring fractions, mean/quantiles.

(Parametric model fits / mixture models can also be placed here if needed, but KM/RMST are usually sufficient for a clean story.)


In [ ]:
# --- ED: QC table + full KM curves (ON/OFF) ---

def qc_table(runs: pd.DataFrame, groups) -> pd.DataFrame:
    df = runs[runs["group"].isin(groups)].copy()

    qc_cells = (df[["group","cell_uid"]]
                .drop_duplicates()
                .groupby("group")["cell_uid"].nunique()
                .rename("n_cells")
                .reset_index())

    qc_runs = (df.groupby(["group","kind"])
               .agg(
                   n_runs=("length_min","size"),
                   frac_left=("left_censored","mean"),
                   frac_right=("right_censored","mean"),
                   mean_len=("length_min","mean"),
                   p90_len=("length_min", lambda x: np.percentile(x, 90)),
               )
               .reset_index())

    out = qc_runs.merge(qc_cells, on="group", how="left")
    return out.sort_values(["kind","group"])

for ds in datasets:
    gene = ds["gene"]
    runs = ds["runs"]

    qc = qc_table(runs, groups=GROUP_ORDER_FULL)
    display(qc)
    qc.pipe(export_transcription_states).to_csv(TABLEDIR / f"{gene}_QC_runs_cells_FULL.csv", index=False)

    # Full KM (exclude left-censored; include right-censored)
    plot_km_panel(runs, kind="ON",  gene=gene, groups=GROUP_ORDER_FULL, exclude_left=True, logy=True,
                  outname=f"ED_{gene}_KM_ON_FULL.pdf", title=f"$\\it{{{gene}}}$: KM active (exclude left-censored)")
    plot_km_panel(runs, kind="OFF", gene=gene, groups=GROUP_ORDER_FULL, exclude_left=True, logy=True,
                  outname=f"ED_{gene}_KM_OFF_FULL.pdf", title=f"$\\it{{{gene}}}$: KM inactive (exclude left-censored)")